# AutoML (Automated Machine Learning)

## 가. AutoML 개요 및 필요성
### 1. **개념** : 머신러닝 모델을 구축할 때 필요한 반복적이고 복잡한 과정(전처리, 모델 선택, 성능 고도화 등)을 자동화하는 기술
### 2. **주요 기능** : 데이터 전처리, 모델 선택, 성능 고도화, 모델 결합 등 복잡한 과정 자동화
### 3. * **필요성** : 고성능 머신러닝 모델 개발 시간을 획기적으로 단축하여 데이터 분석가가 **'비즈니스 가치 창출'**에 더 집중하게 함
### 4. **예시** : AutoGluon (Amazon 개발 오픈소스 라이브러리)

## 나. AutoGluon 소개
* 파이썬 기반의 오픈 소스 머신러닝 라이브러리
* **특징**: 단 몇 줄의 코드로 전문가 수준(SOTA)의 성능 구현

### 1. 전통적 방식 vs AutoGluon 비교
| 구분 | 전통적 방식 (Manual) | AutoGluon (AutoML) |
| :--- | :--- | :--- |
| **모델 탐색** | 모델별 개별 코드 작성 및 파이프라인 구축 | 한 줄의 코드로 수십 개 모델 동시 학습 및 비교 |
| **데이터 전처리** | 수동 처리 (인코딩, 누락 데이터, 스케일링 직접 구현) | 기초 전처리 자동화 (단, 도메인 파생 변수 생성 및 노이즈 정제는 제외) |
| **성능 고도화** | GridSearchCV 등을 통한 개별 하이퍼파라미터 수동 최적화 | 모델별 하이퍼파라미터 최적화 자동 및 다층 스태킹 앙상블 파이프라인 자동화 |
| **성과 (결과)** | 분석가의 숙련도와 시간 투자에 따라 성능 편차 큰 편 | 짧은 시간 내에 전문가 수준(SOTA) 성능 구현 |

## 다. AutoGluon 3-Step 워크플로우
### 1. **TabularPredictor 클래스**: 모델 생성
### 2. **fit 함수**: 학습 및 최적화, best 모델 생성
### 3. **evaluate 함수**: best 모델 성능 평가 (예측 및 평가)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor


In [2]:
# 1. 데이터 불러오기
df = pd.read_csv('./실습데이터/Telco-Customer-Churn.csv')

In [3]:
# 2. 데이터 전처리
# TotalCharges 컬럼 내 공백 문자열을 NaN으로 바꾼 뒤 숫자로 변환하고, 결측치는 0으로 채움
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').dropna(ignore_index=True)

# 모델 학습에 불필요한 고유 식별자(customerID) 컬럼 제거
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# 학습용(Train)과 평가용(Test) 데이터 분할 (stratify로 이탈(Churn) 비율 유지)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=0, stratify=df['Churn'])


In [4]:
# 3. AutoGluon 모델 생성
# label: 예측할 타겟 컬럼명 ('Churn')
predictor = TabularPredictor(
    label='Churn', 
    eval_metric='accuracy', 
    path='autogluon_telecom_model'
)

In [5]:
# 4. 모델 학습 및 최적화 (fit 함수)
predictor.fit(
    train_data=train_df,
    presets='best_quality',
    num_gpus=0,               # CPU 환경인 경우 0으로 설정
    dynamic_stacking=False,   # 사전 탐색(DyStack) 비활성화
    num_stack_levels=1, 
    time_limit=600            # 학습 제한 시간 (초 단위)
)
print()

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.13.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       7.27 GB / 15.73 GB (46.2%)
Disk Space Avail:   153.48 GB / 236.79 GB (64.8%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "c:\Users\playdata2\work\SKN\머신러닝\autogluon_telecom_model"
Train Data Rows:    5634
Train Data Columns: 19
Label Column:       Churn
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  ['No', 'Yes']
	If 'binary' is not the correct problem_type,

In [6]:
# 5. 베스트 모델 성능 평가 (evaluate 함수)
performance = predictor.evaluate(test_df)
print("성능 평가 결과:", performance)

성능 평가 결과: {'accuracy': 0.8034066713981547, 'balanced_accuracy': np.float64(0.7159136118215402), 'mcc': 0.4663574409112717, 'roc_auc': np.float64(0.8484073471285747), 'f1': 0.5884101040118871, 'precision': 0.6622073578595318, 'recall': 0.5294117647058824}


In [8]:
model_path = './autogluon_telecom_model/'

loaded_predictor = TabularPredictor.load(model_path)

performance = loaded_predictor.evaluate(test_df)
print(performance)

{'accuracy': 0.8034066713981547, 'balanced_accuracy': np.float64(0.7159136118215402), 'mcc': 0.4663574409112717, 'roc_auc': np.float64(0.8484073471285747), 'f1': 0.5884101040118871, 'precision': 0.6622073578595318, 'recall': 0.5294117647058824}
